# Safety Jacket Detection - Training with Roboflow Dataset (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anzonathan/Traffic-Police-Agent/blob/main/notebooks/train_jacket_model_colab.ipynb)

> ⚠️ **Before running:** Set runtime to GPU — `Runtime → Change runtime type → T4 GPU`

> 🔑 **Add your Roboflow API key** in the Secrets sidebar (🔑 icon) as `ROBOFLOW_API_KEY`

**Dataset:** Manually annotated on Roboflow (`score-board-pro/traffiic-agent`)

**Pipeline:**
1. Install dependencies
2. Download annotated dataset from Roboflow
3. Fix dataset paths & Train **YOLOv8**
4. Download `best.pt` to your local machine

## Step 1: Verify GPU & Install Dependencies

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found! Go to Runtime → Change runtime type → T4 GPU"
print(f"GPU ready: {torch.cuda.get_device_name(0)}")

In [ ]:
!pip install -q ultralytics roboflow
print("Dependencies installed!")

## Step 2: Download Annotated Dataset from Roboflow

> 🔑 Make sure you added `ROBOFLOW_API_KEY` in the Secrets sidebar (🔑 icon on the left)

In [ ]:
from roboflow import Roboflow
from google.colab import userdata

# Uses Colab Secrets — add ROBOFLOW_API_KEY in the 🔑 sidebar
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace("score-board-pro").project("traffiic-agent")
version = project.version(1)
dataset = version.download("yolov8")

print(f"\nDataset downloaded to: {dataset.location}")

## Step 3: Fix Dataset Paths & Train YOLOv8

Roboflow's `data.yaml` uses relative paths like `../train/images` which don't resolve correctly.
We fix them to use absolute paths before training.

> ⏳ Training may take 15-40 min on a T4 GPU depending on dataset size.

In [ ]:
import os, yaml
from ultralytics import YOLO

data_yaml_path = f"{dataset.location}/data.yaml"
dataset_root = os.path.abspath(dataset.location)

# Fix the paths in data.yaml to use absolute paths
with open(data_yaml_path) as f:
    data_cfg = yaml.safe_load(f)

data_cfg['path'] = dataset_root
data_cfg['train'] = 'train/images'
data_cfg['val'] = 'valid/images'
if 'test' in data_cfg:
    data_cfg['test'] = 'test/images'

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print("Fixed data.yaml:")
with open(data_yaml_path) as f:
    print(f.read())

# Verify the paths exist
for split in ['train', 'valid', 'test']:
    p = os.path.join(dataset_root, split, 'images')
    exists = os.path.exists(p)
    count = len(os.listdir(p)) if exists else 0
    print(f"  {split}: {'✅' if exists else '❌'} {count} images")

# Train
model = YOLO("yolov8n.pt")
model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    device="cuda",
    patience=20,
    augment=True,
    fliplr=0.5,
    mosaic=1.0,
    project="runs/detect",
    name="safety_jacket_v2"
)
print("Training complete!")

## Step 4: Evaluate & Download `best.pt`

After downloading, place it in your local project at `models/best_jacket.pt`.

In [ ]:
import os, glob
from google.colab import files

best_model_path = "runs/detect/safety_jacket_v2/weights/best.pt"

# If the exact path doesn't exist, search for it
if not os.path.exists(best_model_path):
    candidates = glob.glob("runs/detect/*/weights/best.pt")
    if candidates:
        best_model_path = candidates[-1]  # take the latest one
        print(f"Found model at: {best_model_path}")
    else:
        print("ERROR: No best.pt found! Check runs/ directory:")
        !ls -R runs/

if os.path.exists(best_model_path):
    # Evaluate
    trained_model = YOLO(best_model_path)
    metrics = trained_model.val(data=data_yaml_path)
    print(f"\nmAP50:    {metrics.box.map50:.4f}")
    print(f"mAP50-95: {metrics.box.map:.4f}")
    
    # Download
    print("\nDownloading best.pt...")
    files.download(best_model_path)
    print("Done! Move the downloaded file to: models/best_jacket.pt")